In [ ]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import ConcatDataset

import numpy as np
import matplotlib.pyplot as plt

from ViTGSOM import AutoEncoder, ViTLossReconstruction, SomLoss
from help_functions import get_grid_coords, decay_exponential, calculate_QE_TE_Purity, plot_umap_som_weights, get_node_hits, plot_som_weights, plot_som_mnist, plot_som_pie_grid, generate_extended_u_matrix, visualize_u_matrix_extended 

import optuna
import copy

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(), 
])


train_set = datasets.USPS(root="./data", train=True, download=True, transform=transform)
test_set = datasets.USPS(root="./data", train=False, download=True, transform=transform)
dataset_complete = ConcatDataset([train_set, test_set])

In [ ]:
config_usps = {
    'img_size': 16,
    'patch_size': 4,
    'num_of_channels': 1,
    'embed_dim': 16,
    'enc_depth': 4,
    'dec_depth': 2,
    'num_heads': 2,
    'mlp_dim': 64,
    'epochs_phase1': 20,
    'epochs_phase2': 500,
    'epochs_phase3': 100,
    'lr': 0.0005,
    'grow_after_epochs': 10,
    'som_rows': 5,
    'som_cols': 5,
    'stop_growth_purity': 0.90
}

In [ ]:
def phase_1(model, device, loader, config):
    criterionViT = ViTLossReconstruction()
    
    model.som_weights.requires_grad = False
    for param in model.encoder.parameters(): param.requires_grad = True
    for param in model.decoder.parameters(): param.requires_grad = True

    paramsViT = list(model.encoder.parameters()) + list(model.decoder.parameters())
    optimizerViT = optim.AdamW(paramsViT, lr=config['lr'])
    scheduler = CosineAnnealingLR(optimizerViT, T_max=config['epochs_phase1'])


    for epoch in range(config['epochs_phase1']):
        running_mse = 0.0

        for images, _ in loader:
            images = images.to(device)

            reconstructed, latent = model(images)
            som_weights = model.get_som_weights()

            l_nn = criterionViT(images, reconstructed)

            optimizerViT.zero_grad()
            l_nn.backward()
            optimizerViT.step()
            running_mse += l_nn.item()

        scheduler.step()
        
def phase_2(model, device, loader, config):
    criterionSOM = SomLoss()
    
    model.som_weights.requires_grad = True
    for param in model.encoder.parameters(): param.requires_grad = False
    for param in model.decoder.parameters(): param.requires_grad = False

    paramsSOM = model.som_weights
    optimizerSOM = optim.AdamW([paramsSOM], lr=config['lr'])
    scheduler = CosineAnnealingLR(optimizerSOM, T_max=config['epochs_phase2'])

    sigma_start = model.get_sigma()
    sigma_end = 0.3
    beta = (sigma_end / sigma_start) ** (1 / config['grow_after_epochs'])
    epochs_since_reset = 0

    rows, cols = model.get_som_shape() 
    grid_coords = get_grid_coords(rows, cols, device)

    unique_labels = set()

    for epoch in range(config['epochs_phase2']):

        running_som = 0.0

        sigma_t = decay_exponential(sigma_start, beta, epochs_since_reset)

        for images, labels in loader:
            images = images.to(device)
            unique_labels.update(labels.tolist())

            reconstructed, latent = model(images)
            som_weights = model.get_som_weights()

            l_som = criterionSOM(latent, som_weights, grid_coords, sigma_t)

            optimizerSOM.zero_grad()
            l_som.backward()
            optimizerSOM.step()

            running_som += l_som.item()

        # updating learning rule through CosineAnnealingLR
        scheduler.step()

        metrics = calculate_QE_TE_Purity(model, loader, device)

        if epoch > 0 and (epoch + 1) % config['grow_after_epochs'] == 0:

            if metrics["Purity"] > config['stop_growth_purity']:
                break

            epochs_since_reset += 1
            model.start_growth(loader, device)
            paramsSOM = model.get_som_weights()
            optimizerSOM = optim.AdamW([paramsSOM], lr=config['lr'])

            for param_group in optimizerSOM.param_groups: param_group['initial_lr'] = config['lr']

            scheduler = CosineAnnealingLR(optimizerSOM, T_max=config['epochs_phase2'], last_epoch=epoch)
            grid_coords = get_grid_coords(model.current_row_num, model.current_col_num, device)

            sigma_start = model.get_sigma()
            sigma_start = max(sigma_start, 2.0)
            beta = (sigma_end / sigma_start) ** (1 / max(1, config['grow_after_epochs']))
            epochs_since_reset = 0

        else:
            epochs_since_reset += 1
            
def phase_3(model, device, loader, config):
    criterionSOM = SomLoss()
    
    model.som_weights.requires_grad = True
    for param in model.encoder.parameters():
        param.requires_grad = False
    for param in model.decoder.parameters():
        param.requires_grad = False

    paramsSOM = model.get_som_weights()
    optimizerSOM = optim.AdamW([paramsSOM], lr=config['lr'])
    scheduler = CosineAnnealingLR(optimizerSOM, T_max=config['epochs_phase3'], last_epoch=-1)

    sigma_start = 2
    sigma_end = 0.01
    beta = (sigma_end / sigma_start) ** (1 / config['epochs_phase3'])

    grid_coords = get_grid_coords(model.current_row_num, model.current_col_num, device)

    best_purity = 0.0

    for epoch in range(config['epochs_phase3']):

        running_som = 0.0

        sigma_t = decay_exponential(sigma_start, beta, epoch)

        for images, _ in loader:
            images = images.to(device)

            reconstructed, latent = model(images)
            som_weights = model.get_som_weights()

            l_som = criterionSOM(latent, som_weights, grid_coords, sigma_t)

            optimizerSOM.zero_grad()
            l_som.backward()
            optimizerSOM.step()

            running_som += l_som.item()

        # updating learning rule through CosineAnnealingLR
        scheduler.step()

        metrics = calculate_QE_TE_Purity(model, loader, device)

        if metrics["Purity"] > best_purity: best_purity = metrics["Purity"]

    return best_purity, model.current_col_num * model.current_row_num

def objective(trial, dataset, config):
    loader = torch.utils.data.DataLoader(dataset=dataset, batch_size=32, shuffle=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    trial_config = copy.deepcopy(config)
    
    trial_config['lr'] = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
    trial_config['enc_depth'] = trial.suggest_int('enc_depth', 2, 10)
    trial_config['dec_depth'] = trial.suggest_int('dec_depth', 1, 6)
    trial_config['mlp_dim'] = trial.suggest_categorical('mlp_dim', [32, 64, 128, 256])
    trial_config['num_heads'] = trial.suggest_categorical('num_heads', [2, 4, 8])
    head_dim = trial.suggest_categorical('head_dim', [4, 8, 16]) 
    trial_config['embed_dim'] = trial_config['num_heads'] * head_dim
    
    autoencoder = AutoEncoder(img_size=trial_config['img_size'], 
                          patch_size=trial_config['patch_size'], 
                          num_of_channels=trial_config['num_of_channels'], 
                          embed_dim=trial_config['embed_dim'], 
                          enc_depth=trial_config['enc_depth'],                                      
                          dec_depth=trial_config['dec_depth'], 
                          num_heads=trial_config['num_heads'], 
                          mlp_dim=trial_config['mlp_dim'],
                          som_rows=trial_config['som_rows'],
                          som_cols=trial_config['som_cols'])
    
    autoencoder.to(device)
    autoencoder.train()
    
    try:
        phase_1(autoencoder, device, loader, trial_config)
        phase_2(autoencoder, device, loader, trial_config)
        final_purity, grid_size = phase_3(autoencoder, device, loader, trial_config)
    except Exception as e:
        raise optuna.exceptions.TrialPruned()
    
    trial.set_user_attr("final_purity", final_purity)
    trial.set_user_attr("final_grid", grid_size)
    
    score = final_purity - grid_size * 0.0005
    return score

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(lambda trial: objective(trial, dataset_complete, config_usps), n_trials=50)

print("\n==========================================")
print(f"Best Trial score: {study.best_trial.value}")
best_purity = study.best_trial.user_attrs.get("final_purity")
best_grid = study.best_trial.user_attrs.get("final_grid")
print(f"Purity Achieved: {best_purity:.6f}")
print(f"Grid Size:  {best_grid} nodes")
print("Hyperparameters:: ")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")